In [ ]:
import numpy as np
import gymnasium as gym
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import random
import os
import time

In [ ]:
config = dict(
    total_timestep=200_000,
    learning_rate=0.0005,
    gamma=0.99,
    update_frequency=3,
    torch_deterministic=True,
    num_envs=16,
    # ---- 평가 / 다중 seed 설정 ----
    eval_interval=5_000,      # 이 step마다 greedy 평가
    eval_episodes=10,         # 평가 1회당 에피소드 수
    seeds=list(range(10)),    # 돌릴 seed 목록
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
class CriticNetwork(nn.Module):
  def __init__(self):
    super().__init__()
    self.fc1 = nn.Linear(4,128)
    self.fc2 = nn.Linear(128,64)
    self.fc3 = nn.Linear(64,1)

  def forward(self,a):
    a = self.fc1(a)
    a = torch.relu(a)
    a = self.fc2(a)
    a = torch.relu(a)
    a = self.fc3(a)
    return a


class ActorNetwork(nn.Module):
  def __init__(self):
    super().__init__()
    self.fc1 = nn.Linear(4,128)
    self.fc2 = nn.Linear(128,64)
    self.fc3 = nn.Linear(64,2)

  def forward(self,a):
    a = self.fc1(a)
    a = torch.relu(a)
    a = self.fc2(a)
    a = torch.relu(a)
    a = self.fc3(a)
    return a


In [ ]:
# ---- 저장 경로 설정 ----
# 결과/체크포인트를 저장할 기본 폴더. 본인 환경에 맞게 바꿔서 사용하세요.
# Colab에서 Google Drive에 저장하려면 아래 두 줄의 주석을 풀고 BASE_DIR을 Drive 안의 폴더로 지정하면 됩니다.
# from google.colab import drive
# drive.mount('/content/drive')
BASE_DIR = './A2C'


In [ ]:
#   ActorNetwork : obs(N,4) -> logits(N,2)
#   CriticNetwork: obs(N,4) -> value(N,1)

save_dir = os.path.join(BASE_DIR, 'A2C_checkpoints_AdvNormalize_3step')
results_dir = os.path.join(BASE_DIR, 'A2C_results_AdvNormalize_3step')
os.makedirs(save_dir, exist_ok=True)
os.makedirs(results_dir, exist_ok=True)
log_interval = 1000      # env step 기준
print_interval = 10_000  # env step 기준


def make_env(env_id, seed):
    def thunk():
        env = gym.make(env_id)
        env.action_space.seed(seed)
        return env
    return thunk


def evaluate(actor_net, n_episodes, seed):
    """greedy 정책으로 n_episodes 돌려서 평균 return 반환"""
    env = gym.make('CartPole-v1')
    returns = []
    for ep in range(n_episodes):
        obs, _ = env.reset(seed=seed + ep)
        done, ep_ret = False, 0.0
        while not done:
            with torch.no_grad():
                logit = actor_net(torch.as_tensor(obs, dtype=torch.float32, device=device).unsqueeze(0))
                action = torch.argmax(logit, dim=1).item()
            obs, r, term, trunc, _ = env.step(int(action))
            ep_ret += r
            done = term or trunc
        returns.append(ep_ret)
    env.close()
    return float(np.mean(returns))


def train(seed, config):
    # ---------- seed 고정 (네트워크 생성 전에) ----------
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.backends.cudnn.deterministic = config['torch_deterministic']

    envs = gym.vector.SyncVectorEnv(
        [make_env('CartPole-v1', seed=seed + i) for i in range(config['num_envs'])]
    )

    actor_net = ActorNetwork().to(device)
    critic_net = CriticNetwork().to(device)
    actor_optimizer = optim.Adam(actor_net.parameters(), lr=config['learning_rate'])
    critic_optimizer = optim.Adam(critic_net.parameters(), lr=config.get('critic_learning_rate', config['learning_rate']))
    ent_coef = config.get('ent_coef', 0.01)
    max_grad_norm = config.get('max_grad_norm', 0.5)
    gamma = config['gamma']
    n_steps = config['update_frequency']   # rollout 길이 (예: 5)
    # total_timestep = 전체 env step 수. 한 반복에 num_envs step씩 진행하므로 나눠줌
    num_iterations = config['total_timestep'] // config['num_envs']

    obs, _ = envs.reset(seed=seed)
    autoreset = np.zeros(config['num_envs'], dtype=bool)
    ep_returns = np.zeros(config['num_envs'])
    episode_returns = []
    eval_steps, eval_returns = [], []
    last_actor_loss = float('nan')
    last_critic_loss = float('nan')
    # 간격은 전부 env step 기준 (num_envs 배수로 증가하므로 % 대신 임계값으로 체크)
    next_eval_step = 0
    next_log_step = 0
    next_print_step = print_interval
    start_time = time.time()

    # rollout 버퍼 (각 원소 shape: (N,))
    buf = {k: [] for k in ['values', 'log_probs', 'entropy',
                           'rewards', 'terminated', 'done', 'valid', 'next_values']}

    def do_eval(step):
        ret = evaluate(actor_net, config['eval_episodes'], seed=seed * 100_000 + step)
        eval_steps.append(step)
        eval_returns.append(ret)
        return ret

    try:
        for global_timestep in range(num_iterations):
            env_steps = global_timestep * config['num_envs']

            # 주기적 평가 (step 0 포함)
            if env_steps >= next_eval_step:
                do_eval(env_steps)
                next_eval_step += config['eval_interval']

            # ---------- 행동 선택 (gradient 유지) ----------
            obs_t = torch.as_tensor(obs, dtype=torch.float32, device=device)
            values = critic_net(obs_t).squeeze(-1)             # (N,)
            dist = torch.distributions.Categorical(logits=actor_net(obs_t))
            actions = dist.sample()

            next_obs, rewards, terminated, truncated, info = envs.step(actions.cpu().numpy())
            done = terminated | truncated
            valid = ~autoreset   # autoreset 직후 step은 reset transition → loss에서 제외

            # done step의 bootstrap용으로 매 step V(next_obs) 계산
            # (next-step autoreset이라 done step의 next_obs는 실제 마지막 상태)
            with torch.no_grad():
                next_values = critic_net(
                    torch.as_tensor(next_obs, dtype=torch.float32, device=device)).squeeze(-1)

            buf['values'].append(values)
            buf['log_probs'].append(dist.log_prob(actions))
            buf['entropy'].append(dist.entropy())
            buf['rewards'].append(torch.as_tensor(rewards, dtype=torch.float32, device=device))
            buf['terminated'].append(torch.as_tensor(terminated, dtype=torch.float32, device=device))
            buf['done'].append(torch.as_tensor(done, device=device))
            buf['valid'].append(torch.as_tensor(valid, dtype=torch.float32, device=device))
            buf['next_values'].append(next_values)

            # ---------- episode return 기록 ----------
            ep_returns[valid] += rewards[valid]
            for i in np.where(done)[0]:
                episode_returns.append(ep_returns[i])
                ep_returns[i] = 0

            autoreset = done
            obs = next_obs

            # ---------- n-step 업데이트 ----------
            if len(buf['values']) == n_steps:
                B = {k: torch.stack(v) for k, v in buf.items()}   # (T, N)
                for v in buf.values():
                    v.clear()

                # n-step return (뒤에서부터)
                #  - 에피소드가 끝난 step: 그 step의 next_obs로 bootstrap
                #    (terminated면 (1-term)=0 이라 reward만, truncated면 V(final obs) 사용)
                #  - 마지막 step이 안 끝났으면: V(next_obs_{T-1})로 bootstrap
                returns = torch.zeros_like(B['rewards'])
                R = B['next_values'][-1]
                for t in reversed(range(n_steps)):
                    R = torch.where(B['done'][t], B['next_values'][t], R)
                    R = B['rewards'][t] + gamma * (1.0 - B['terminated'][t]) * R
                    returns[t] = R

                mask = B['valid']
                n_valid = mask.sum()
                if n_valid > 0:
                    # advantage 정규화 추가
                    m = mask.bool()
                    raw_adv = (returns - B['values']).detach()
                    adv_valid = raw_adv[m]
                    advantages = (raw_adv - adv_valid.mean()) / (adv_valid.std() + 1e-8)

                    critic_loss = (((B['values'] - returns) ** 2) * mask).sum() / n_valid
                    entropy_mean = (B['entropy'] * mask).sum() / n_valid
                    actor_loss = -((B['log_probs'] * advantages) * mask).sum() / n_valid \
                                 - ent_coef * entropy_mean

                    critic_optimizer.zero_grad()
                    critic_loss.backward()
                    torch.nn.utils.clip_grad_norm_(critic_net.parameters(), max_grad_norm)
                    critic_optimizer.step()

                    actor_optimizer.zero_grad()
                    actor_loss.backward()
                    torch.nn.utils.clip_grad_norm_(actor_net.parameters(), max_grad_norm)
                    actor_optimizer.step()

                    # 업데이트가 n_steps마다라서 % 대신 다음 로그 시점으로 체크
                    if env_steps >= next_log_step:
                        next_log_step += log_interval
                        last_actor_loss = actor_loss.item()
                        last_critic_loss = critic_loss.item()

            if env_steps >= next_print_step:
                next_print_step += print_interval
                sps = int(env_steps / (time.time() - start_time))
                print(f"[seed {seed}] step {env_steps:>7d} | eval return {eval_returns[-1]:6.1f} "
                      f"| actor {last_actor_loss:.4f} | critic {last_critic_loss:.4f} | SPS {sps}")

        # 마지막 평가 + 최종 모델 저장
        do_eval(num_iterations * config['num_envs'])
        torch.save({'actor_net': actor_net.state_dict(),
                    'critic_net': critic_net.state_dict(),
                    'config': config, 'seed': seed},
                   os.path.join(save_dir, f'final_seed{seed}.pt'))
    finally:
        envs.close()

    return np.array(eval_steps), np.array(eval_returns)

In [ ]:
for seed in config['seeds']:
    path = os.path.join(results_dir, f'seed{seed}.npz')
    if os.path.exists(path):
        print(f"seed {seed}: 이미 있음, 건너뜀")
        continue
    steps, rets = train(seed, config)
    np.savez(path, steps=steps, returns=rets)
    print(f"seed {seed} 완료 | 최종 eval return {rets[-1]:.1f}")

In [ ]:
import glob
import matplotlib.pyplot as plt
from scipy.stats import trim_mean

def iqm_ci(x, reps=10_000, alpha=0.05, seed=0):
    """x: (num_seeds, ...) -> (IQM, CI 하한, CI 상한). seed 축으로 bootstrap"""
    x = np.asarray(x, dtype=float)
    rng = np.random.default_rng(seed)
    n = x.shape[0]
    idx = rng.integers(0, n, size=(reps, n))
    boot = trim_mean(x[idx], 0.25, axis=1)
    lo, hi = np.percentile(boot, [100 * alpha / 2, 100 * (1 - alpha / 2)], axis=0)
    return trim_mean(x, 0.25, axis=0), lo, hi

files = sorted(glob.glob(os.path.join(results_dir, 'seed*.npz')))
data = [np.load(f) for f in files]
steps = data[0]['steps']
R = np.stack([d['returns'] for d in data])          # (n_seeds, n_eval_points)
n_seeds = R.shape[0]
print(f"seed {n_seeds}개 로드")

# ---------- 1. 학습 곡선 ----------
m, lo, hi = iqm_ci(R)
fig, ax = plt.subplots(figsize=(7, 4))
for r in R:                                          # 개별 seed (흐리게)
    ax.plot(steps, r, color='gray', alpha=0.2, lw=0.8)
ax.plot(steps, m, lw=2, label=f'A2C IQM (n={n_seeds} seeds)')
ax.fill_between(steps, lo, hi, alpha=0.25, label='95% bootstrap CI')
ax.axhline(475, ls='--', c='k', lw=0.8, label='threshold 475')
ax.set_xlabel('Environment steps')
ax.set_ylabel('Evaluation return (greedy)')
ax.set_ylim(0, 520)
ax.legend(loc='lower right')
ax.set_title('CartPole-v1 A2C')
plt.tight_layout()
plt.savefig(os.path.join(results_dir, 'learning_curve.png'), dpi=200)
plt.show()

# ---------- 2. seed별 요약 지표 ----------
T = config['total_timestep']
avg_return = np.trapezoid(R, steps, axis=1) / (steps[-1] - steps[0])   # 정규화 AUC = 학습 중 평균 성능
final_return = R[:, -3:].mean(axis=1)                                   # 마지막 3개 평가 지점 평균
solved = R >= 475
reached = solved.any(axis=1)
steps_to_475 = np.where(reached, steps[solved.argmax(axis=1)], T)       # 도달 못 하면 T로 처리

for name, v in [('Normalized AUC', avg_return),
                ('Final return (last 3 evals)', final_return),
                ('Steps to 475', steps_to_475)]:
    mm, l, h = iqm_ci(v)
    print(f"{name:28s} IQM = {mm:9.1f}   95% CI = [{l:9.1f}, {h:9.1f}]")
print(f"475 도달한 seed: {reached.sum()}/{n_seeds}")

In [ ]:
import os, glob
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import trim_mean

base = BASE_DIR   # 경로 설정 셀에서 지정한 폴더

# 비교할 설정: {그래프 라벨: 결과 폴더}
runs = {
    '3step' : f'{base}/A2C_results_AdvNormalize_3step',
    '5step' : f'{base}/A2C_results_AdvNormalize',
    '8step' : f'{base}/A2C_results_AdvNormalize_8step',
    '10step': f'{base}/A2C_results_AdvNormalize_10step',
    '16step': f'{base}/A2C_results_AdvNormalize_16step',
}

def iqm_ci(x, reps=10_000, alpha=0.05, seed=0):
    x = np.asarray(x, dtype=float)
    rng = np.random.default_rng(seed)
    n = x.shape[0]
    idx = rng.integers(0, n, size=(reps, n))
    boot = trim_mean(x[idx], 0.25, axis=1)
    lo, hi = np.percentile(boot, [100 * alpha / 2, 100 * (1 - alpha / 2)], axis=0)
    return trim_mean(x, 0.25, axis=0), lo, hi

def load_results(folder):
    files = sorted(glob.glob(os.path.join(folder, 'seed*.npz')))
    data = [np.load(f) for f in files]
    steps = data[0]['steps']
    R = np.stack([d['returns'] for d in data])      # (n_seeds, n_points)
    return steps, R

# ---------- 학습 곡선 ----------
fig, ax = plt.subplots(figsize=(7, 4))
results = {}
for label, folder in runs.items():
    steps, R = load_results(folder)
    results[label] = (steps, R)
    m, lo, hi = iqm_ci(R)
    ax.plot(steps, m, lw=2, label=f'{label} (n={len(R)})')
    ax.fill_between(steps, lo, hi, alpha=0.2)

ax.axhline(475, ls='--', c='k', lw=0.8)
ax.set_xlabel('Environment steps')
ax.set_ylabel('IQM evaluation return (95% CI)')
ax.set_ylim(0, 520)
ax.legend(loc='upper left')
ax.set_title('CartPole-v1')
plt.tight_layout()
plt.savefig(f'{base}/learning_curve_compare.png', dpi=200)
plt.show()

# ---------- 요약 지표 ----------
for label, (steps, R) in results.items():
    T = steps[-1]
    auc = np.trapezoid(R, steps, axis=1) / (steps[-1] - steps[0])
    final = R[:, -3:].mean(axis=1)
    solved = R >= 475
    reached = solved.any(axis=1)
    to475 = np.where(reached, steps[solved.argmax(axis=1)], T)

    print(f"=== {label} ===")
    for name, v in [('Normalized AUC', auc), ('Final return', final), ('Steps to 475', to475)]:
        mm, l, h = iqm_ci(v)
        print(f"  {name:15s} IQM = {mm:9.1f}   95% CI = [{l:9.1f}, {h:9.1f}]")
    print(f"  475 도달: {reached.sum()}/{len(R)}\n")

In [ ]:
labels = list(results.keys())
fig, axes = plt.subplots(1, len(labels), figsize=(4 * len(labels), 3.5), sharey=True)

curves = {lb: iqm_ci(R) for lb, (steps, R) in results.items()}
for ax, lb in zip(axes, labels):
    steps, R = results[lb]
    for other in labels:                       # 다른 설정은 회색 참고선
        if other != lb:
            ax.plot(results[other][0], curves[other][0], color='gray', alpha=0.4, lw=1)
    m, lo, hi = curves[lb]
    ax.plot(steps, m, lw=2, color='C0')
    ax.fill_between(steps, lo, hi, alpha=0.25, color='C0')
    ax.axhline(475, ls='--', c='k', lw=0.8)
    ax.set_title(f'{lb}', fontsize=10)
    ax.set_xlabel('Environment steps')
axes[0].set_ylabel('IQM eval return (95% CI)')
axes[0].set_ylim(0, 520)
plt.tight_layout()
plt.show()